### Обучение классификатора символов по жестам
#### Кочетков Александр, Падии 3

Качаем данные, распаковываем

In [ ]:
!apt-get update
!apt-get install -y aria2

In [ ]:
!wget -O bukva.zip "https://rndml-team-cv.obs.ru-moscow-1.hc.sbercloud.ru/datasets/bukva/bukva.zip"


In [ ]:
!unzip bukva.zip -d burka/


In [4]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split


In [5]:
DATA_DIR = "burka/trimmed"
ANNOTATIONS_FILE = "burka/annotations.tsv"
FRAMES_DIR = "cache_burka"
N_FRAMES = 16
SIZE = 224

Так как не хочется, чтобы random search очень долго работал, оставим только первые 10 символов для классификации. Так же птичка нашептали, что для ускорения обучения можно предобработать все видео (из каждого равномерно выбрать несколько кадров, отресайзить их и поместить в массив np, чтобы при обучении пользоваться готовыми данными). Этот трюк уменьшил время обучения примерно в 5 раз. Из минусов, модель на каждой эпохе видит одни и те же кадры, из-за чего может переобучаться, а так же мы почти лишаемся аугментаций изображений. С другой стороны, модель, обучаемая без этого трюка, так же очень сильно переобучалась не меньше (не знаю, почему :(  )

In [7]:
df = pd.read_csv(ANNOTATIONS_FILE, sep='\t')
df['video_path'] = df['attachment_id'].apply(lambda x: os.path.join(DATA_DIR, f"{x}.mp4"))

allowed_classes = {0: "no_event",
                   1:"Ё", 2:"А", 3:"Б", 4:"В", 5:"Г", 6:"Д", 7:"Е", 8:"Ж", 9:"З"}

df_filtered = df[df['text'].isin(allowed_classes.values())].copy()

df_train = df_filtered[df_filtered['train'] == True].copy()
df_test  = df_filtered[df_filtered['train'] == False].copy()
df_train, df_val = train_test_split(df_train, test_size=0.10, random_state=42, 
                                    stratify=df_train['text'])

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

Train: 837, Val: 93, Test: 200


In [13]:
import os
import cv2
import numpy as np
from tqdm import tqdm

def extract_frames(df, split_name):
    split_dir = os.path.join(FRAMES_DIR, split_name)
    os.makedirs(split_dir, exist_ok=True)
    
    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        out_path = os.path.join(split_dir, f"{row['attachment_id']}.npy")
        cap = cv2.VideoCapture(row["video_path"])
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total <= 0:
            np.save(out_path, np.zeros((N_FRAMES, SIZE, SIZE, 3), dtype=np.uint8))
            cap.release()
            continue

        indices = np.linspace(0, total - 1, N_FRAMES, dtype=int)
        frames = []
        for fi in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fi))
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (SIZE, SIZE))
            else:
                frame = np.zeros((SIZE, SIZE, 3), dtype=np.uint8)
            frames.append(frame)
        cap.release()
        np.save(out_path, np.stack(frames).astype(np.uint8))

In [14]:
extract_frames(df_val, "val")

Processing val: 100%|██████████| 93/93 [00:44<00:00,  2.08it/s]


In [15]:
extract_frames(df_train, "train")

Processing train: 100%|██████████| 837/837 [06:18<00:00,  2.21it/s]


In [16]:
extract_frames(df_test, "test")

Processing test: 100%|██████████| 200/200 [01:35<00:00,  2.10it/s]


Класс датасета:

In [34]:
import torch
from torch.utils.data import Dataset
import numpy as np
import os
import torchvision.transforms as T


def video_augment_tensor(video: np.ndarray, size=224, n_frames=16):
    video = torch.tensor(video, dtype=torch.float32).permute(0,3,1,2) / 255.0
    T_aug = T.Compose([
        T.RandomHorizontalFlip(),
        T.RandomResizedCrop(size, scale=(0.8, 1.0)),
        T.ColorJitter(0.2, 0.2, 0.2)
    ])
    augmented_frames = torch.stack([T_aug(f) for f in video])
    if augmented_frames.shape[0] > n_frames:
        idx = torch.linspace(0, augmented_frames.shape[0]-1, n_frames*2, dtype=torch.int)
        idx = idx[torch.randperm(len(idx))[:n_frames]]
        augmented_frames = augmented_frames[idx]

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)
    augmented_frames = (augmented_frames - mean) / std
    return augmented_frames



class RSLVideoDataset(Dataset):
    def __init__(self, df, split_dir, allowed_classes=None, transform=None, n_frames=16, size=224):
        if allowed_classes is not None:
            df = df[df['text'].isin(allowed_classes.values())].copy()
        self.df = df.reset_index(drop=True)
        self.split_dir = split_dir
        self.transform = transform
        self.n_frames = n_frames
        self.size = size

        self.classes = sorted(df['text'].unique())
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.split_dir, f"{row['attachment_id']}.npy")
        video = np.load(path)

        if self.transform:
            video = self.transform(video, size=self.size, n_frames=self.n_frames)
        else:
            video = torch.tensor(video, dtype=torch.float32).permute(0,3,1,2) / 255.0
            mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
            std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)
            video = (video - mean) / std

        label = self.class_to_idx[row['text']]
        return video, label

In [35]:
train_dir = os.path.join(FRAMES_DIR, "train")
val_dir = os.path.join(FRAMES_DIR, "val")
test_dir = os.path.join(FRAMES_DIR, "test")

train_dataset = RSLVideoDataset(df_train, train_dir, transform=video_augment_tensor)
val_dataset = RSLVideoDataset(df_val, val_dir)
test_dataset = RSLVideoDataset(df_test, test_dir)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=4)

Архитектуру возьмем такую: MobileNetV2 в качестве backbone, TSM заменим на простое усреднение признаков по кадрам, и так сработает.

In [44]:
import torch
import torch.nn as nn
import torchvision.models as models

class TSMMobileNetV2(nn.Module):
    def __init__(self, num_classes=10, n_frames=16, pretrained=True, dropout = 0.4):
        super().__init__()
        self.n_frames = n_frames
        self.backbone = models.mobilenet_v2(pretrained=pretrained).features
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)       
        x = self.backbone(x)         
        x = self.avgpool(x)       
        x = x.view(B, T, -1)   
        x = x.mean(dim=1)            
        x = self.fc(x)               
        return x

In [61]:
model = TSMMobileNetV2(num_classes=10, n_frames=N_FRAMES, pretrained=True, dropout=0.5).to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Так как данных мало и модель постоянно переобучается, заморозим первые 10 слоев.

In [58]:
for name, param in model.backbone.named_parameters():
    layer_idx = int(name.split('.')[0])
    if layer_idx < 10:
        param.requires_grad = False

И обучаем:

In [62]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

num_epochs = 20
device = "cuda" if torch.cuda.is_available() else "cpu"

criterion = nn.CrossEntropyLoss(label_smoothing=0.2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=4e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

best_val_acc = 0.0
best_model_path = "best_model.pth"

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for videos, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Train"):
        videos = videos.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * videos.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    train_loss /= total
    train_acc = correct / total

    
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for videos, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Val"):
            videos = videos.to(device)
            labels = labels.to(device)
            outputs = model(videos)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * videos.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    val_loss /= total
    val_acc = correct / total
    
    print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f} Acc {train_acc:.4f} | Val Loss {val_loss:.4f} Acc {val_acc:.4f}")

    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc
        }, best_model_path)
        print(f"  -> Saved new best model with val_acc: {best_val_acc:.4f}")
    
    scheduler.step()

Epoch 1/20 - Val: 100%|██████████| 12/12 [00:00<00:00, 13.28it/s]


Epoch 1: Train Loss 2.2528 Acc 0.1876 | Val Loss 1.8761 Acc 0.3871
  -> Saved new best model with val_acc: 0.3871


Epoch 2/20 - Val: 100%|██████████| 12/12 [00:00<00:00, 12.89it/s]


Epoch 2: Train Loss 1.6800 Acc 0.5639 | Val Loss 1.5216 Acc 0.6452
  -> Saved new best model with val_acc: 0.6452


Epoch 3/20 - Val: 100%|██████████| 12/12 [00:00<00:00, 13.07it/s]


...

Epoch 18: Train Loss 0.9378 Acc 0.9988 | Val Loss 1.4174 Acc 0.7419


Epoch 19/20 - Val: 100%|██████████| 12/12 [00:00<00:00, 12.64it/s]


Epoch 19: Train Loss 0.9448 Acc 0.9988 | Val Loss 1.4282 Acc 0.7527


Epoch 20/20 - Val: 100%|██████████| 12/12 [00:00<00:00, 13.63it/s]

Epoch 20: Train Loss 0.9396 Acc 1.0000 | Val Loss 1.4053 Acc 0.7527


Как мы можем видеть, модель даже так очень быстро начинает переобучаться. Возьмем лучшую и проверим на тесте:

In [63]:
model = TSMMobileNetV2(num_classes=10)
model.to(device)

checkpoint = torch.load("best_model.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for videos, labels in test_loader:
        videos = videos.to(device)
        labels = labels.to(device)
        
        outputs = model(videos)
        preds = torch.argmax(outputs, dim=1)
        
        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(accuracy)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


0.72


Попробуем сделать Random Search. Будем пробовать разный dropout, label_smoothing и разное число замороженных слоев у backbone. Сложно сказать, что мы сможем подобрать хорошие гиперпараметры, так как для моделей большим числом замороженных слоев нужно сильно больше эпох для хорошего обучения, а модели с малым числом замороженных слоев быстро переобучатся. Поэтому как поиск гиперпараметров это не очень хорошо работает. Попробуем поставить задачу так: у нас мало времени, мало данных, и мы хотим получить максимально хорошую еще не переобученную модель. При такой постановке задачи все выглядит куда лучше.

In [68]:
search_space = {
    "freeze_layers": [0, 5, 10, 15, 19],    
    "dropout": [0.3, 0.4, 0.5, 0.6],
    "label_smoothing": [0.0, 0.1, 0.2, 0.3]
}

N_TRIALS = 12                                    
N_EPOCHS_PER_TRIAL = 10                      

best_overall_val_acc = 0.0
best_config = None
best_model_checkpoint = "best_random_search_model.pth"

print(f"Запускаем Random Search ({N_TRIALS} проб)...")

for trial in range(N_TRIALS):
    config = {k: random.choice(v) for k, v in search_space.items()}
    
    print(f"\n=== Trial {trial+1}/{N_TRIALS} ===")
    print(f"Конфигурация: freeze_layers={config['freeze_layers']}, "
          f"dropout={config['dropout']}, label_smoothing={config['label_smoothing']}")
    

    model = TSMMobileNetV2(
        num_classes=len(train_dataset.classes),
        n_frames=N_FRAMES,
        pretrained=True,
        dropout=config["dropout"]
    ).to(device)
    

    freeze_k = config["freeze_layers"]
    for name, param in model.backbone.named_parameters():
        try:
            layer_idx = int(name.split('.')[0])
            param.requires_grad = layer_idx >= freeze_k
        except:
            param.requires_grad = True
    

    criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])
    
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4,
        weight_decay=4e-5
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS_PER_TRIAL, eta_min=1e-6)
    

    trial_best_val_acc = 0.0
    for epoch in range(N_EPOCHS_PER_TRIAL):
        # ------------------- Train -------------------
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        for videos, labels in tqdm(train_loader, desc=f"Trial {trial+1} Epoch {epoch+1}/{N_EPOCHS_PER_TRIAL} - Train", leave=False):
            videos = videos.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(videos)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * videos.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        train_loss /= total
        train_acc = correct / total
        

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for videos, labels in tqdm(val_loader, desc=f"Trial {trial+1} Epoch {epoch+1}/{N_EPOCHS_PER_TRIAL} - Val", leave=False):
                videos = videos.to(device)
                labels = labels.to(device)
                outputs = model(videos)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * videos.size(0)
                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)
        
        val_loss /= total
        val_acc = correct / total
        
        print(f"  Epoch {epoch+1:2d} | Train Loss {train_loss:.4f} Acc {train_acc:.4f} | "
              f"Val Loss {val_loss:.4f} Acc {val_acc:.4f}")
        
        if val_acc > trial_best_val_acc:
            trial_best_val_acc = val_acc
        
        scheduler.step()
    

    print(f"  Trial finished. Best val_acc for this trial: {trial_best_val_acc:.4f}")
    
    if trial_best_val_acc > best_overall_val_acc:
        best_overall_val_acc = trial_best_val_acc
        best_config = config
        torch.save({
            'trial': trial + 1,
            'config': config,
            'val_acc': best_overall_val_acc,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict()
        }, best_model_checkpoint)
        print(f"  >>> НОВАЯ ЛУЧШАЯ МОДЕЛЬ! val_acc = {best_overall_val_acc:.4f} <<<")

print(f"\nRandom Search завершён. Лучшая конфигурация: {best_config}")
print(f"Лучшая val_acc после 10 эпох: {best_overall_val_acc:.4f}")
print(f"Модель сохранена в {best_model_checkpoint}")


Запускаем Random Search (12 проб)...

=== Trial 1/12 ===
Конфигурация: freeze_layers=15, dropout=0.6, label_smoothing=0.0


  Epoch  1 | Train Loss 2.3280 Acc 0.1386 | Val Loss 2.1725 Acc 0.1828


  Epoch  2 | Train Loss 1.9581 Acc 0.2927 | Val Loss 1.8050 Acc 0.3226


  Epoch  3 | Train Loss 1.4906 Acc 0.4922 | Val Loss 1.5772 Acc 0.3441


  Epoch  4 | Train Loss 1.1801 Acc 0.6141 | Val Loss 1.4139 Acc 0.4516


...

  Epoch  6 | Train Loss 2.2679 Acc 0.1637 | Val Loss 2.3153 Acc 0.1613


  Epoch  7 | Train Loss 2.2495 Acc 0.1840 | Val Loss 2.3156 Acc 0.1505


  Epoch  8 | Train Loss 2.2506 Acc 0.1935 | Val Loss 2.3134 Acc 0.1505


  Epoch  9 | Train Loss 2.2401 Acc 0.1888 | Val Loss 2.3108 Acc 0.1613


  Epoch 10 | Train Loss 2.2402 Acc 0.1804 | Val Loss 2.3108 Acc 0.1505
  Trial finished. Best val_acc for this trial: 0.1720

Random Search завершён. Лучшая конфигурация: {'freeze_layers': 5, 'dropout': 0.3, 'label_smoothing': 0.0}
Лучшая val_acc после 10 эпох: 0.7957
Модель сохранена в best_random_search_model.pth


Как мы видим, сложно сказать, что в каком то из вариантов обучение идет стабильно и одновременно не слишком медленно. Посмотрим, какой результат не тестовой выборке:

In [69]:
model = TSMMobileNetV2(num_classes=10)
model.to(device)

checkpoint = torch.load("best_random_search_model.pth", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for videos, labels in test_loader:
        videos = videos.to(device)
        labels = labels.to(device)
        
        outputs = model(videos)
        preds = torch.argmax(outputs, dim=1)
        
        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(accuracy)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


0.76


Как мы видим, +4 пункта accuracy мы получили.

Сделаем оценку неопределенности лучшей модели:

In [73]:
def evaluate_uncertainty_mc_dropout(model_path="best_random_search_model.pth", n_forward_passes=30):
    checkpoint = torch.load(model_path, map_location=device)
    model = TSMMobileNetV2(
        num_classes=len(test_dataset.classes),
        n_frames=N_FRAMES,
        pretrained=False,
        dropout=checkpoint['config']['dropout'] if 'config' in checkpoint else 0.5
    ).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    model.train()
    
    all_mean_probs = []
    all_std_probs = []
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for videos, labels in tqdm(test_loader, desc="MC Dropout inference"):
            videos = videos.to(device)
            batch_preds = []
            
            for _ in range(n_forward_passes):
                outputs = model(videos)                 
                probs = torch.softmax(outputs, dim=1)
                batch_preds.append(probs)
            
            batch_preds = torch.stack(batch_preds)     
            
            mean_prob = batch_preds.mean(dim=0)          
            std_prob = batch_preds.std(dim=0)        
            
            _, predicted = torch.max(mean_prob, 1)
            
            all_mean_probs.append(mean_prob.cpu())
            all_std_probs.append(std_prob.cpu())
            all_labels.append(labels.cpu())
            all_preds.append(predicted.cpu())
    
    mean_probs = torch.cat(all_mean_probs, dim=0)
    std_probs = torch.cat(all_std_probs, dim=0)
    labels = torch.cat(all_labels, dim=0)
    preds = torch.cat(all_preds, dim=0)
    
    accuracy = (preds == labels).float().mean().item()
    
    avg_uncertainty = std_probs.mean(dim=1).mean().item()
    
    print(f"   Accuracy (mean prediction) : {accuracy:.4f}")
    print(f"   Средняя неопределённость   : {avg_uncertainty:.4f}")
    
    return mean_probs, std_probs, labels, accuracy, avg_uncertainty

mean_probs, std_probs, true_labels, test_acc, avg_unc = evaluate_uncertainty_mc_dropout()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
MC Dropout inference: 100%|██████████| 25/25 [00:12<00:00,  1.94it/s]

   Accuracy (mean prediction) : 0.2700
   Средняя неопределённость   : 0.0165


### Результаты

На бейзлайне был получен результат 0.72 accuracy, после применения random search удалось добиться 0.76 accuracy. Учитывая простоту модели не самое эффективное использование данных и их малое количество, результат удовлетворительный.